# GenAIScope v0.5.0 — What's New: Smart Memory Compaction + Automatic Observability

This notebook is a focused smoke test and live demo for the **two features added in v0.5.0**:

1. **Semantic memory compaction** — finds memories that say the same thing in different words
   (not just exact-text duplicates) and merges them, cutting the tokens/$ spent every time that
   memory gets injected into a future prompt.
2. **Automatic observability** — `OpenAIAdapter`/`AnthropicAdapter`/`GeminiAdapter`, the MCP
   server, and the REST API now record latency, token usage, cost, and success/error status for
   every call automatically, with zero extra code at the call site.

> This is not a replacement for the full `genaiscope_complete_colab_testsv0.2.91.ipynb` smoke
> test (which covers the whole package from v0.1 onward) — it's a targeted "does the new stuff
> actually work, and why should I care" notebook. Run top-to-bottom; cells are defensive and will
> mark unavailable features as skipped rather than breaking the whole notebook.


In [ ]:
# Runtime controls
INSTALL_SOURCE = "github"  # "github" or "pypi"
GITHUB_REPO = "https://github.com/TravelXML/GenAIScope.git"

# Workspace used by the tests
WORKSPACE = "/content/genaiscope_v050_workspace"


## 1. Clean workspace and install GenAIScope v0.5.0

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

workspace = Path(WORKSPACE)
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print("Workspace:", workspace)
print("Python:", sys.version)


def run_cmd(command, title=None, check=False, cwd=None):
    print("\n" + "=" * 100)
    if title:
        print(title)
        print("=" * 100)
    print("$", command)
    print("-" * 100)

    result = subprocess.run(command, shell=True, text=True, capture_output=True, cwd=cwd)

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    print("Exit code:", result.returncode)

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {command}")

    return result


run_cmd("python -m pip install --upgrade pip", "Upgrade pip", check=True)

if INSTALL_SOURCE == "github":
    run_cmd(
        f'python -m pip install --upgrade "git+{GITHUB_REPO}"',
        "Install GenAIScope from GitHub (main)",
        check=True,
    )
else:
    run_cmd("python -m pip install --upgrade genaiscope", "Install GenAIScope from PyPI", check=True)

# v0.5.0 demos need the REST API extra (fastapi/uvicorn) for the observability section.
# providers (openai/anthropic/google) is NOT required -- the adapter demos below use mock
# clients, exactly like GenAIScope's own test suite, so no API keys are needed.
run_cmd('python -m pip install --upgrade "genaiscope[server]"', "Install REST API extra", check=True)


## 2. Confirm the installed version and new v0.5.0 exports

In [ ]:
import genaiscope

print("GenAIScope version:", genaiscope.__version__)
assert genaiscope.__version__.startswith("0.5"), "This notebook expects a 0.5.x install"

from genaiscope.memory import compact_memories, find_semantic_duplicates, CompactionReport
from genaiscope.adapters import openai_summarizer, anthropic_summarizer, gemini_summarizer
from genaiscope.tracing import LocalTracer

print("New v0.5.0 exports imported OK:")
print("-", compact_memories)
print("-", find_semantic_duplicates)
print("-", CompactionReport)


In [ ]:
TEST_RESULTS = []


def record(name, status, details=""):
    TEST_RESULTS.append({"test": name, "status": status, "details": str(details)[:500]})
    icon = "✅" if status == "PASS" else ("⚠️" if status == "SKIP" else "❌")
    print(f"{icon} {name}: {status}")
    if details:
        print(details)


## Part A — Semantic Memory Compaction

**The problem**: imagine an AI assistant remembers a user preference twice, written two
slightly different ways — maybe the user said it once, then said it again differently a week
later. Every time that memory gets pulled into a prompt, you're now paying to send the *same
fact* to GPT/Claude/Gemini twice. GenAIScope's old dedupe (`v0.3.0`) only caught exact-text
duplicates. v0.5.0 understands *meaning*, not just spelling.

### A.1 — Seed two paraphrased memories + one unrelated one

In [ ]:
from pathlib import Path
from genaiscope.embeddings.factory import get_embedder
from genaiscope.vector.local_vector import LocalVectorStore
from genaiscope.memory.factory import MemoryStore

embedder = get_embedder("local")  # zero-dependency, always available
vector_store = LocalVectorStore(db_path=Path(WORKSPACE) / "compaction_vec.db")
memory = MemoryStore(db_path=Path(WORKSPACE) / "compaction_mem.db", embedder=embedder, vector_store=vector_store)

mem_a = memory.add("User prefers concise CTO-level answers", memory_type="preference", importance=8, tags=["style"])
mem_b = memory.add("User prefers concise CTO level answer", memory_type="preference", importance=3, tags=["tone"])
mem_c = memory.add("Redis is the production memory backend for high-traffic apps", memory_type="general")

print("Seeded 3 memories:")
for m in (mem_a, mem_b, mem_c):
    print(f"  [{m.importance}] {m.content!r}  tags={m.tags}")

record("Seed compaction demo memories", "PASS", f"{memory.stats().total_memories} memories stored")


### A.2 — Old text-based dedupe (v0.3.0) misses the paraphrase

In [ ]:
from genaiscope.memory.dedupe import find_duplicates

old_dedupe_groups = find_duplicates(memory)
print("Text-based duplicate groups found:", old_dedupe_groups)

if not old_dedupe_groups:
    print("\nAs expected: exact-text dedupe sees these as two unrelated memories, "
          "even though they mean the same thing.")
    record("Text-based dedupe misses paraphrase (expected)", "PASS", old_dedupe_groups)
else:
    record("Text-based dedupe misses paraphrase (expected)", "FAIL", old_dedupe_groups)


### A.3 — New semantic compaction (v0.5.0) catches it

In [ ]:
from genaiscope.memory.compaction import find_semantic_duplicates

semantic_groups = find_semantic_duplicates(memory, threshold=0.5)
print("Semantic duplicate groups found:")
for group in semantic_groups:
    for item in group:
        print(f"  - {item.content!r}")

assert len(semantic_groups) == 1, "Expected exactly one semantic cluster"
assert {i.id for i in semantic_groups[0]} == {mem_a.id, mem_b.id}, "Expected the two paraphrased memories, not the Redis one"
record("Semantic compaction finds the paraphrase cluster", "PASS", semantic_groups)


### A.4 — Preview the savings with `dry_run=True` (no changes made yet)

This is the "show me before you touch anything" mode. It tells you exactly how many tokens and
dollars get saved on every *future* prompt that injects this memory — without deleting or
merging anything yet.

In [ ]:
from genaiscope.memory.compaction import compact_memories

preview = compact_memories(memory, strategy="keep_best", threshold=0.5, dry_run=True)
print("Dry-run compaction report:")
for k, v in preview.model_dump().items():
    print(f"  {k}: {v}")

assert preview.dry_run is True
assert preview.merged_ids == [] and preview.deleted_ids == [], "dry_run must not modify the store"
assert memory.stats().total_memories == 3, "dry_run must not delete anything"

print(f"\n→ Merging this one cluster would save ~{preview.tokens_saved} tokens per future "
      f"injection (~${preview.dollar_savings['gpt-4']:.6f} on GPT-4 pricing alone).")
record("Dry-run preview computes savings without modifying the store", "PASS", preview.model_dump())


### A.5 — Apply it: deterministic merge (`keep_best`, no LLM call, zero extra cost)

`keep_best` just keeps whichever memory has the highest importance (newest as a tie-break) and
unions the tags from every memory in the cluster. No API key, no extra cost — this is the
default-safe path.

In [ ]:
applied = compact_memories(memory, strategy="keep_best", threshold=0.5, dry_run=False)
print("Applied compaction report:")
for k, v in applied.model_dump().items():
    print(f"  {k}: {v}")

survivor = memory.get(applied.merged_ids[0])
print(f"\nSurvivor memory: {survivor.content!r}")
print(f"Tags merged from both originals: {survivor.tags}")
print(f"Total memories: 3 -> {memory.stats().total_memories}")

assert memory.stats().total_memories == 2, "Should have merged 2 memories into 1"
assert set(survivor.tags) == {"style", "tone"}, "Tags from both originals should be unioned"
assert survivor.content == "User prefers concise CTO-level answers", "Higher-importance memory should win"
record("keep_best merge: tags unioned, higher-importance memory survives", "PASS", survivor.model_dump())


### A.6 — LLM-assisted merge (`synthesize`): combine facts instead of picking one

Sometimes neither memory is strictly "better" — each has information worth keeping. The
`synthesize` strategy calls a summarizer (any callable that takes a list of strings and returns
one merged sentence) to write a single memory preserving facts from *all* originals.

This demo uses a tiny stub summarizer so it runs with no API key. GenAIScope ships ready-made
factories for real providers — `openai_summarizer()`, `anthropic_summarizer()`,
`gemini_summarizer()` — so the exact same `compact_memories(..., summarizer=openai_summarizer())`
call works identically no matter which provider does the merging.

In [ ]:
vector_store2 = LocalVectorStore(db_path=Path(WORKSPACE) / "compaction_vec2.db")
memory2 = MemoryStore(db_path=Path(WORKSPACE) / "compaction_mem2.db", embedder=embedder, vector_store=vector_store2)

memory2.add("User is based in Bangalore and works east-coast US hours", memory_type="general")
memory2.add("User lives in Bangalore but keeps US east-coast working hours", memory_type="general")


def stub_summarizer(texts):
    """Toy 'LLM': just concatenates facts. A real summarizer would call an actual model."""
    return "; ".join(texts)


synth_report = compact_memories(memory2, strategy="synthesize", summarizer=stub_summarizer, threshold=0.5, dry_run=False)
merged = memory2.get(synth_report.merged_ids[0])

print("Synthesis used:", synth_report.synthesis_used)
print("Merged content:", merged.content)

assert synth_report.synthesis_used is True
assert "Bangalore" in merged.content
record("synthesize merge combines facts via a summarizer callable", "PASS", merged.content)
memory2.close()


### A.7 — The proof: compaction does not hurt search quality

This is the most important check, not just a demo. GenAIScope's own retrieval-quality harness
(recall@k / precision@k / MRR) is used to prove that after merging, you can still find the same
information just as reliably as before — compaction saves tokens without losing recall.

In [ ]:
from genaiscope.evals.memory_eval import compute_metrics

vector_store3 = LocalVectorStore(db_path=Path(WORKSPACE) / "compaction_vec3.db")
memory3 = MemoryStore(db_path=Path(WORKSPACE) / "compaction_mem3.db", embedder=embedder, vector_store=vector_store3)

a3 = memory3.add("User prefers concise CTO-level answers", memory_type="preference", importance=8)
b3 = memory3.add("User prefers concise CTO level answer", memory_type="preference", importance=5)
c3 = memory3.add("Redis is the production memory backend for high-traffic apps", memory_type="general")

queries = {
    "how should I answer the user": [a3.id, b3.id],
    "Redis backend usage": [c3.id],
}


def recall_at_5(store):
    results_per_query = [[r.item.id for r in store.search(q, limit=5, mode="hybrid")] for q in queries]
    expected_per_query = list(queries.values())
    recall, precision, mrr = compute_metrics(results_per_query, expected_per_query, top_k=5)
    return recall


recall_before = recall_at_5(memory3)

report3 = compact_memories(memory3, strategy="keep_best", threshold=0.5, dry_run=False)
queries["how should I answer the user"] = [i.id for i in memory3.list(limit=10) if i.source == "compaction"]

recall_after = recall_at_5(memory3)

print(f"Recall@5 before compaction: {recall_before:.3f}")
print(f"Recall@5 after  compaction: {recall_after:.3f}")
print(f"Memories merged: {report3.memories_merged}  |  Total memories: 3 -> {memory3.stats().total_memories}")

assert recall_after >= recall_before, "Compaction must never reduce retrieval recall"
record("Compaction preserves retrieval recall (eval harness)", "PASS",
       f"recall_before={recall_before:.3f} recall_after={recall_after:.3f}")
memory3.close()


### A.8 — Same thing from the CLI: `genaiscope memory compact`

In [ ]:
cli_db = str(Path(WORKSPACE) / "compaction_cli.db")
run_cmd(f'genaiscope memory add "User prefers concise CTO-level answers" --type preference --embedder local --db-path {cli_db}',
        "Add memory #1")
run_cmd(f'genaiscope memory add "User prefers concise CTO level answer" --type preference --embedder local --db-path {cli_db}',
        "Add memory #2 (paraphrased)")

result = run_cmd(f"genaiscope memory compact --embedder local --threshold 0.5 --db-path {cli_db}",
                  "Dry-run preview via CLI (default)")
record("CLI memory compact (dry-run)", "PASS" if result.returncode == 0 else "FAIL", result.stdout)

result = run_cmd(f"genaiscope memory compact --embedder local --threshold 0.5 --apply --db-path {cli_db}",
                  "Apply compaction via CLI")
record("CLI memory compact (--apply)", "PASS" if result.returncode == 0 else "FAIL", result.stdout)

result = run_cmd(f"genaiscope memory stats --db-path {cli_db}", "Memory stats after CLI compaction")
record("CLI memory stats reflects merge", "PASS" if result.returncode == 0 else "FAIL", result.stdout)


## Part B — Automatic Observability

**The problem**: GenAIScope has shipped a local tracing system since early versions — but
nothing actually *called* it. Wiring up `OpenAIAdapter`/`AnthropicAdapter`/`GeminiAdapter`, the
MCP server, or the REST API gave you memory injection, but **zero visibility** into latency,
token usage, cost, or errors. v0.5.0 wires tracing into all three, opt-in and off by default.

### B.1 — Adapter tracing: pass `tracer=...`, get latency/tokens/cost/status for free

This uses a mock OpenAI client (same pattern GenAIScope's own test suite uses) so no API key is
needed for the demo — the same `tracer=` argument works identically with a real `OpenAI()`
client, and with `AnthropicAdapter`/`GeminiAdapter`.

In [ ]:
from types import SimpleNamespace
from unittest.mock import MagicMock
from genaiscope.adapters.openai_adapter import OpenAIAdapter

obs_memory = MemoryStore(db_path=Path(WORKSPACE) / "obs_mem.db")
tracer = LocalTracer(db_path=Path(WORKSPACE) / "obs_traces.db")


def mock_openai_client(reply="Sure, here's a concise answer."):
    msg = SimpleNamespace(content=reply)
    choice = SimpleNamespace(message=msg)
    completion = SimpleNamespace(choices=[choice])
    client = MagicMock()
    client.chat.completions.create.return_value = completion
    return client


client = mock_openai_client()
adapter = OpenAIAdapter(obs_memory, client, user_id="demo", store_user_turns=False, tracer=tracer)
adapter.chat(messages=[{"role": "user", "content": "How should you answer me?"}], model="gpt-4o-mini")

traces = tracer.list()
print(f"{len(traces)} trace(s) recorded automatically -- no manual instrumentation written:")
for t in traces:
    print(f"  name={t.name!r} provider={t.provider} status={t.status} latency_ms={t.latency_ms:.2f}")

assert len(traces) == 1 and traces[0].status == "success"
record("OpenAIAdapter auto-records a success trace", "PASS", traces[0].model_dump())


### B.2 — Errors are traced too, and the real exception still propagates

Observability shouldn't swallow errors. The failing call still raises normally to your code —
GenAIScope just leaves a trace record behind on the way out.

In [ ]:
client.chat.completions.create.side_effect = RuntimeError("rate limited by provider")

try:
    adapter.chat(messages=[{"role": "user", "content": "another question"}], model="gpt-4o-mini")
    raised = False
except RuntimeError as exc:
    raised = True
    print("Exception still propagated to caller, as expected:", exc)

error_trace = tracer.list()[0]  # most recent
print(f"\nMost recent trace: status={error_trace.status!r} error={error_trace.error!r}")

assert raised, "The real exception must still propagate to the caller"
assert error_trace.status == "error" and "rate limited" in (error_trace.error or "")
record("Adapter errors are traced AND still raise normally", "PASS", error_trace.model_dump())


### B.3 — Same tracer, same pattern, across Anthropic and Gemini

The point of a provider-agnostic tracer is that switching providers doesn't change how you get
observability -- same `tracer=` argument, same trace shape, different `provider` field.

In [ ]:
from genaiscope.adapters.anthropic_adapter import AnthropicAdapter
from genaiscope.adapters.gemini_adapter import GeminiAdapter

# Anthropic
content_block = SimpleNamespace(text="A concise reply.")
anthropic_msg = SimpleNamespace(content=[content_block])
anthropic_client = MagicMock()
anthropic_client.messages.create.return_value = anthropic_msg

anthropic_adapter = AnthropicAdapter(obs_memory, anthropic_client, user_id="demo", store_user_turns=False, tracer=tracer)
anthropic_adapter.chat(messages=[{"role": "user", "content": "hello"}])

# Gemini
gemini_model = MagicMock()
gemini_model.generate_content.return_value = SimpleNamespace(text="A concise reply.")
gemini_client = MagicMock()
gemini_client.GenerativeModel.return_value = gemini_model

gemini_adapter = GeminiAdapter(obs_memory, gemini_client, user_id="demo", store_user_turns=False, tracer=tracer)
gemini_adapter.chat(messages=[{"role": "user", "content": "hello"}])

providers_seen = {t.provider for t in tracer.list()}
print("Providers traced so far:", providers_seen)
assert {"openai", "anthropic", "gemini"}.issubset(providers_seen)
record("Anthropic + Gemini adapters traced with the same tracer", "PASS", providers_seen)


### B.4 — MCP tool calls are traced too (no live MCP transport needed for this demo)

`genaiscope serve mcp --trace` wires the same tracer into every MCP tool call (`memory_remember`,
`memory_search`, etc.). This demo calls the dispatch function directly, the same way the MCP
server does internally, without needing the optional `mcp` SDK installed.

In [ ]:
from genaiscope.mcp.server import _dispatch

mcp_result = _dispatch(obs_memory, "memory_remember", {"content": "Sapan likes bullet points"}, tracer)
print("MCP tool result:", mcp_result)

mcp_traces = [t for t in tracer.list() if t.provider == "mcp"]
print(f"\n{len(mcp_traces)} MCP trace(s):")
for t in mcp_traces:
    print(f"  name={t.name!r} status={t.status}")

assert len(mcp_traces) == 1 and mcp_traces[0].name == "mcp.memory_remember"
record("MCP tool dispatch is traced", "PASS", mcp_traces[0].model_dump())


### B.5 — REST API requests are traced via one middleware (every route, zero route edits)

`genaiscope serve api --trace` adds a single FastAPI middleware that traces every request by
HTTP status code -- 4xx/5xx is `status="error"`, otherwise `"success"`. This demo drives the
ASGI app directly with `httpx` (in-process, no real network socket) so it runs anywhere,
including Colab.

In [ ]:
import httpx
from genaiscope.server.app import create_app

rest_memory = MemoryStore(db_path=Path(WORKSPACE) / "obs_rest_mem.db")
rest_tracer = LocalTracer(db_path=Path(WORKSPACE) / "obs_rest_traces.db")
app = create_app(rest_memory, tracer=rest_tracer)

# Jupyter/Colab cells support top-level `await` directly -- asyncio.run() would fail here
# because IPython already runs its own event loop.
transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    health_resp = await client.get("/health")
    missing_resp = await client.get("/v1/memory/does-not-exist")

print("GET /health ->", health_resp.status_code)
print("GET /v1/memory/does-not-exist ->", missing_resp.status_code)

rest_traces = rest_tracer.list()
for t in rest_traces:
    print(f"  name={t.name!r} status={t.status} metadata={t.metadata}")

assert any(t.status == "success" and t.metadata.get("status_code") == 200 for t in rest_traces)
assert any(t.status == "error" and t.metadata.get("status_code") == 404 for t in rest_traces)
record("REST API middleware traces success and error requests", "PASS", [t.model_dump() for t in rest_traces])
rest_memory.close()
rest_tracer.close()


### B.6 — `--trace` flags exist on the servers, and traces show up via the CLI + dashboard

In [ ]:
import re

# Rich-formatted --help output styles each '-' separately, splitting flags like
# --trace across ANSI escape codes -- strip them before searching for a flag name.
ansi_re = re.compile(r"\x1b\[[0-9;]*m")


def has_flag(stdout, flag):
    return flag in ansi_re.sub("", stdout)


result = run_cmd("genaiscope serve mcp --help", "serve mcp --help")
record("serve mcp --help shows --trace", "PASS" if has_flag(result.stdout, "--trace") else "FAIL", result.stdout)

result = run_cmd("genaiscope serve api --help", "serve api --help")
record("serve api --help shows --trace", "PASS" if has_flag(result.stdout, "--trace") else "FAIL", result.stdout)

trace_db = str(Path(WORKSPACE) / "obs_traces.db")
result = run_cmd(f"genaiscope trace list --db-path {trace_db}", "genaiscope trace list (adapter traces from B.1-B.3)")
record("CLI trace list shows adapter traces", "PASS" if result.returncode == 0 else "FAIL", result.stdout)

result = run_cmd(f"genaiscope trace stats --db-path {trace_db}", "genaiscope trace stats")
record("CLI trace stats", "PASS" if result.returncode == 0 else "FAIL", result.stdout)


In [ ]:
try:
    from genaiscope.dashboard import generate_dashboard

    dashboard_path = Path(WORKSPACE) / "dashboard.html"
    generated = generate_dashboard(output_path=dashboard_path, db_path=str(Path(WORKSPACE) / "obs_traces.db"))
    html_text = Path(generated).read_text(encoding="utf-8", errors="ignore")
    print("Dashboard generated:", generated, f"({len(html_text)} chars)")

    from IPython.display import HTML, display
    display(HTML(html_text[:200000]))

    record("Dashboard renders the traces recorded above", "PASS", generated)
except Exception as exc:
    record("Dashboard rendering", "FAIL", exc)


## 3. Known limitation worth knowing about

Trace `estimated_cost` will show **$0.00** for most real model names. `CostAnalyzer`'s pricing
table matches short aliases (`"gpt-4"`, `"claude-3-sonnet"`, `"gemini-pro"`), not real provider
model strings like `"gpt-4o-mini-2024-07-18"`. Latency, token counts, and success/error status
are unaffected -- this is a pricing-lookup gap, not a tracing bug. A model-name normalizer is a
natural fast-follow.

## 4. Final summary

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(TEST_RESULTS)
    display(df)
except Exception:
    print(TEST_RESULTS)

passed = sum(1 for r in TEST_RESULTS if r["status"] == "PASS")
failed = sum(1 for r in TEST_RESULTS if r["status"] == "FAIL")
skipped = sum(1 for r in TEST_RESULTS if r["status"] == "SKIP")

print("\n" + "=" * 80)
print("GENAISCOPE v0.5.0 FEATURE VALIDATION SUMMARY")
print("=" * 80)
print("PASS:", passed)
print("FAIL:", failed)
print("SKIP:", skipped)
print("=" * 80)

if failed == 0:
    print("✅ Every v0.5.0 compaction + observability check passed.")
    print("   - Paraphrased duplicate memories: found and merged automatically.")
    print("   - Retrieval recall after compaction: did not regress.")
    print("   - Every OpenAI/Anthropic/Gemini/MCP/REST call above: automatically traced.")
else:
    print("❌ Some checks failed. Inspect details above.")


## Notes

- This notebook focuses on what changed in **v0.5.0** (semantic memory compaction +
  automatic observability). For a from-scratch smoke test of the whole package (prompt
  inspection, PII, memory, file memory, tracing, dashboard, embeddings, vector search, MCP, REST,
  eval harness, etc.), see `genaiscope_complete_colab_testsv0.2.91.ipynb` in this repo.
- Adapter/MCP demos above use mock clients so the notebook runs with no API keys. Swap in a real
  `OpenAI()`/`Anthropic()`/`google.generativeai` client and the exact same `tracer=` argument
  records real token usage and cost.
- For release validation, also run `pytest`, `ruff check .`, and `mypy src/` from the repository.
